# 007 checks

# Imports

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.geo import LatLon
from uavloc.matching.correlator import CorrelatorConfig, SearchSurface, decide, posterior
from uavloc.matching.replay import load_records

# Params

In [ ]:
EXP = "007-search-prior-and-gate"
BASELINE_EXP = "006-segmentation-ceiling"
BASELINE_RUN = "006.1"
REPRODUCTION_RUN = "007.1"
GRID_CELLS = 101
CELL_M = 0.4
TEMPERATURES = [0.25, 0.5, 1.0, 2.0]
BUMP_WIDTH_M = 1.5
BUMP_CENTRE_M = (3.0, -2.0)
LINE_ANGLE_DEG = 30.0
LINE_WIDTH_M = 1.0
LINE_SHIFT_M = 4.0
S_MAX_M = 3.0
C_MIN = 0.03
SEED = 42

# Data

In [ ]:
side_m = GRID_CELLS * CELL_M
empty = SearchSurface(LatLon(50.0, 25.0), 0.0, np.zeros((GRID_CELLS, GRID_CELLS)), 1.0, (0, 0), (0, 0),
                      (0, 0), (GRID_CELLS - 1, GRID_CELLS - 1), (side_m - CELL_M, side_m - CELL_M))
offsets = empty.offset_grid_m()
config = CorrelatorConfig(1.7, 60.0, matching="weighted-ncc", class_weights=(0.0, 1.0))
gated = replace(config, gate="full", gate_s_max_m=S_MAX_M, gate_c_min=C_MIN)
offsets[:, GRID_CELLS // 2, GRID_CELLS // 2]

In [ ]:
def gaussian_log(centre_m: tuple[float, float], width_m: float) -> np.ndarray:
    """A surface whose posterior at temperature 1 is a Gaussian of `width_m` around `centre_m`."""
    return -((offsets - np.array(centre_m)[:, None, None]) ** 2).sum(0) / (2 * width_m ** 2)

In [ ]:
direction = np.array([np.cos(np.radians(LINE_ANGLE_DEG)), np.sin(np.radians(LINE_ANGLE_DEG))])
normal = np.array([-direction[1], direction[0]])
line = -(np.einsum("i,ihw->hw", normal, offsets) - LINE_SHIFT_M) ** 2 / (2 * LINE_WIDTH_M ** 2)

# Run

In [ ]:
noise = np.random.default_rng(SEED).normal(size=(GRID_CELLS, GRID_CELLS))
{t: posterior(replace(empty, heatmap=noise), replace(config, temperature=t)).map_cell == np.unravel_index(noise.argmax(), noise.shape)
 for t in TEMPERATURES}

In [ ]:
bump = posterior(replace(empty, heatmap=gaussian_log(BUMP_CENTRE_M, BUMP_WIDTH_M)), config)
{"s1_m": bump.axes[0], "s2_m": bump.axes[1], "expected_m": BUMP_WIDTH_M, "mean_m": bump.mean_m.round(3)}

In [ ]:
ridge = posterior(replace(empty, heatmap=line), config)
s1, s2, e1, e2 = ridge.axes
{"s1_m": s1, "s2_m": s2, "|e1 . line|": abs(e1 @ direction), "|e2 . normal|": abs(e2 @ normal)}

In [ ]:
cases = {"bump": gaussian_log(BUMP_CENTRE_M, BUMP_WIDTH_M), "line": line, "flat": np.zeros_like(line)}
decisions = {name: decide(replace(empty, heatmap=z), gated) for name, z in cases.items()}
decisions["low coverage"] = decide(replace(empty, heatmap=cases["bump"], coverage=C_MIN / 2), gated)
pd.DataFrame({name: {"decision": c.decision, "applied_m": np.round(c.offset_m, 2), "map_m": np.round(c.map_offset_m, 2)}
              for name, c in decisions.items()}).T

In [ ]:
{"across move . normal (expect the line's shift)": float(np.dot(decisions["line"].offset_m, normal)),
 "across move . line (expect 0)": float(np.dot(decisions["line"].offset_m, direction))}

# Reproduction

In [ ]:
baseline = load_records(BASELINE_EXP, BASELINE_RUN)
rerun = load_records(EXP, REPRODUCTION_RUN)
same_draws = (baseline[["frame", "noise_m", "bearing_deg"]].values == rerun[["frame", "noise_m", "bearing_deg"]].values).all()
{"same draws": bool(same_draws),
 **{f"max |diff| {c}": float(np.nanmax(np.abs(baseline[c] - rerun[c]))) for c in ["error_m", "corrected_lat", "corrected_lon", "peak_z"]}}

# Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, name in zip(axes, ["bump", "line"]):
    c = decisions[name]
    ax.imshow(np.exp(cases[name]), extent=empty.extent_m(), cmap="viridis")
    ax.plot(*c.map_offset_m, "ws", mfc="none"); ax.plot(*c.offset_m, "rx", ms=10, mew=2)
    s1, s2 = c.uncertainty_m
    ax.set_title(f"{name}: {c.decision}, s1 {s1:.1f} m, s2 {s2:.1f} m", fontsize=9)
fig.tight_layout()